# Notebook 11 — Experiment 19: Model Design as a Fairness Variable
### Novelty 2
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Two controlled comparisons where everything is held constant except one design
decision.

**Part A — loss function.** Logistic Regression minimises log loss. Linear SVM
minimises hinge loss. Both were trained in Experiments 1 and 2 on identical
features. Nothing needs retraining — the comparison reads saved predictions.

**Part B — tree growth.** XGBoost grows trees level-wise, completing each depth
before going deeper. LightGBM grows leaf-wise, always expanding whichever leaf
promises the largest loss reduction. The hypothesis is that leaf-wise growth
gives more modelling capacity to hard examples such as sarcasm posts, because
those yield the biggest loss reduction per split.

One new training run. Fills **Table 8**.

## Cell 1: Setup

In [2]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
!pip install -q lightgbm

Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal


## Cell 2: Part A — Loss Function

No retraining. Experiments 1 and 2 already produced everything needed.

In [4]:
!pip install -q aif360 fairlearn

import importlib, sys
for m in list(sys.modules):
    if m.startswith(("aif360", "fairlearn")):
        del sys.modules[m]

try:
    from aif360.datasets import BinaryLabelDataset
    print("AIF360 OK")
except Exception as e:
    print(f"AIF360 failed: {e}")

try:
    from fairlearn.metrics import demographic_parity_difference
    print("Fairlearn OK")
except Exception as e:
    print(f"Fairlearn failed: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 12.5 MB/s eta 0:00:00
AIF360 OK
Fairlearn OK


In [5]:
D = PATHS["data"]

pred_lr  = load_predictions("exp01", "LogisticRegression")
pred_svm = load_predictions("exp02", "LinearSVM")

print("="*62)
print("PART A — LOSS FUNCTION: log loss vs hinge loss")
print("="*62)
print("Same features, same data, same split. Only the loss differs.\n")

audit_lr  = full_fairness_audit(pred_lr)
audit_svm = full_fairness_audit(pred_svm)

from sklearn.metrics import f1_score
f1_lr  = f1_score(pred_lr["y_true"],  pred_lr["y_pred"],  average="macro")
f1_svm = f1_score(pred_svm["y_true"], pred_svm["y_pred"], average="macro")

aod_lr  = audit_lr["AIF360 AOD"].abs().mean()  if "AIF360 AOD" in audit_lr  else np.nan
aod_svm = audit_svm["AIF360 AOD"].abs().mean() if "AIF360 AOD" in audit_svm else np.nan

print(f"  Logistic Regression : macro F1 {f1_lr:.4f}   mean |AOD| {aod_lr:.4f}")
print(f"  Linear SVM          : macro F1 {f1_svm:.4f}   mean |AOD| {aod_svm:.4f}")
print(f"  Delta AOD           : {aod_svm - aod_lr:+.4f}")

mc = mcnemar_test(pred_lr, pred_svm)
print(f"\n  McNemar test: chi2={mc['statistic']:.3f}  p={mc['p_value']:.4f}")
print(f"  LR right / SVM wrong: {mc['n10']}   LR wrong / SVM right: {mc['n01']}")
print(f"  {'Significant difference' if mc['p_value'] < 0.05 else 'No significant difference'} "
      f"at alpha 0.05")

PART A — LOSS FUNCTION: log loss vs hinge loss
Same features, same data, same split. Only the loss differs.



pip install 'aif360[inFairness]'


  Logistic Regression : macro F1 0.5672   mean |AOD| 0.0312
  Linear SVM          : macro F1 0.5648   mean |AOD| 0.0329
  Delta AOD           : +0.0017

  McNemar test: chi2=9.718  p=0.0018
  LR right / SVM wrong: 389   LR wrong / SVM right: 482
  Significant difference at alpha 0.05


## Cell 3: Part A — Sarcasm Subgroup Detail

The overall AOD can hide a subgroup-specific effect, which is what this study
cares about.

In [6]:
def sarcasm_row(audit, name):
    if audit.empty or "Subgroup" not in audit.columns:
        return {"Model": name}
    s = audit[audit["Subgroup"] == "sarcasm"]
    if s.empty:
        return {"Model": name}
    s = s.iloc[0]
    return {"Model": name,
            "Sarcasm AOD": s.get("AIF360 AOD", np.nan),
            "Sarcasm DIR": s.get("AIF360 DIR", np.nan),
            "Sarcasm SPD": s.get("AIF360 SPD", np.nan)}

sarc = pd.DataFrame([sarcasm_row(audit_lr,  "Logistic Regression"),
                     sarcasm_row(audit_svm, "Linear SVM")])
print("SARCASM SUBGROUP — LOSS FUNCTION EFFECT")
print("="*62)
print(sarc.round(4).to_string(index=False))

SARCASM SUBGROUP — LOSS FUNCTION EFFECT
              Model  Sarcasm AOD  Sarcasm DIR  Sarcasm SPD
Logistic Regression       0.0571       1.1232       0.0705
         Linear SVM       0.0583       1.1098       0.0636


## Cell 4: Part B — Train LightGBM

The only new training in this notebook. Same features as Experiment 5's XGBoost
so the comparison stays controlled.

In [7]:
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder

tw_train = pd.read_parquet(D / "tw_train.parquet")
tw_test  = pd.read_parquet(D / "tw_test.parquet")
X_train  = sp.load_npz(D / "tw_Xtrain_fs1.npz")
X_test   = sp.load_npz(D / "tw_Xtest_fs1.npz")
y_train  = tw_train["sentiment"].values
y_test   = tw_test["sentiment"].values
sg_test  = tw_test["subgroup_primary"].values

print("="*62)
print("PART B — TREE GROWTH: level-wise vs leaf-wise")
print("="*62)
print("Training LightGBM on the same features as Exp 5 XGBoost.\n")

le = LabelEncoder().fit(y_train)

t0 = time.time()
lgbm = LGBMClassifier(
    n_estimators=200, learning_rate=0.1,
    num_leaves=31,            # leaf-wise growth control
    random_state=SEED, n_jobs=-1, verbose=-1)
lgbm.fit(X_train, le.transform(y_train))

y_pred  = le.inverse_transform(lgbm.predict(X_test))
y_proba = lgbm.predict_proba(X_test)
elapsed = time.time() - t0

res_lgbm = evaluate_model(y_test, y_pred, y_proba, label="LightGBM")
print(f"  macro F1 : {res_lgbm['Macro F1']:.4f}")
print(f"  fit time : {elapsed/60:.1f} min")

save_model(lgbm, "exp19", "LightGBM")
save_predictions("exp19", "LightGBM", y_test, y_pred, y_proba, sg_test)

PART B — TREE GROWTH: level-wise vs leaf-wise
Training LightGBM on the same features as Exp 5 XGBoost.

  macro F1 : 0.5301
  fit time : 1.7 min
  saved model -> exp19_LightGBM.pkl
  saved predictions -> exp19_LightGBM_TweetEval.parquet  (12,284 rows)


PosixPath('/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/predictions/exp19_LightGBM_TweetEval.parquet')

## Cell 5: Part B — Compare Against XGBoost

In [8]:
pred_xgb  = load_predictions("exp05", "XGBoost")
pred_lgbm = load_predictions("exp19", "LightGBM")

audit_xgb  = full_fairness_audit(pred_xgb)
audit_lgbm = full_fairness_audit(pred_lgbm)

f1_xgb  = f1_score(pred_xgb["y_true"],  pred_xgb["y_pred"],  average="macro")
f1_lgbm = f1_score(pred_lgbm["y_true"], pred_lgbm["y_pred"], average="macro")

aod_xgb  = audit_xgb["AIF360 AOD"].abs().mean()  if "AIF360 AOD" in audit_xgb  else np.nan
aod_lgbm = audit_lgbm["AIF360 AOD"].abs().mean() if "AIF360 AOD" in audit_lgbm else np.nan

print(f"  XGBoost  (level-wise) : macro F1 {f1_xgb:.4f}   mean |AOD| {aod_xgb:.4f}")
print(f"  LightGBM (leaf-wise)  : macro F1 {f1_lgbm:.4f}   mean |AOD| {aod_lgbm:.4f}")
print(f"  Delta AOD             : {aod_lgbm - aod_xgb:+.4f}")

sarc_b = pd.DataFrame([sarcasm_row(audit_xgb,  "XGBoost (level-wise)"),
                       sarcasm_row(audit_lgbm, "LightGBM (leaf-wise)")])
print("\nSARCASM SUBGROUP — TREE GROWTH EFFECT")
print("="*62)
print(sarc_b.round(4).to_string(index=False))

print("\nHypothesis under test: leaf-wise growth allocates more capacity to")
print("hard examples, so LightGBM should show a smaller sarcasm AOD.")

  XGBoost  (level-wise) : macro F1 0.4519   mean |AOD| 0.0336
  LightGBM (leaf-wise)  : macro F1 0.5301   mean |AOD| 0.0194
  Delta AOD             : -0.0142

SARCASM SUBGROUP — TREE GROWTH EFFECT
               Model  Sarcasm AOD  Sarcasm DIR  Sarcasm SPD
XGBoost (level-wise)      -0.0705       0.7843      -0.1179
LightGBM (leaf-wise)      -0.0302       0.7479      -0.1444

Hypothesis under test: leaf-wise growth allocates more capacity to
hard examples, so LightGBM should show a smaller sarcasm AOD.


## Cell 6: Table 8

In [9]:
def get_sarc(audit, col):
    if audit.empty or "Subgroup" not in audit.columns:
        return np.nan
    s = audit[audit["Subgroup"] == "sarcasm"]
    return s.iloc[0].get(col, np.nan) if not s.empty else np.nan

table8 = pd.DataFrame([
    {"Comparison": "Part A — Loss function",
     "Factor Varied": "log loss vs hinge loss",
     "Model A": "Logistic Regression (Exp 1)", "Model B": "Linear SVM (Exp 2)",
     "A Macro F1": f1_lr, "B Macro F1": f1_svm,
     "A Mean AOD": aod_lr, "B Mean AOD": aod_svm,
     "Delta AOD": aod_svm - aod_lr,
     "A Sarcasm AOD": get_sarc(audit_lr, "AIF360 AOD"),
     "B Sarcasm AOD": get_sarc(audit_svm, "AIF360 AOD"),
     "Delta Sarcasm AOD": get_sarc(audit_svm, "AIF360 AOD") - get_sarc(audit_lr, "AIF360 AOD"),
     "Fairer Model": "Logistic Regression" if aod_lr < aod_svm else "Linear SVM",
     "McNemar p": mc["p_value"]},
    {"Comparison": "Part B — Tree growth",
     "Factor Varied": "level-wise vs leaf-wise",
     "Model A": "XGBoost (Exp 5)", "Model B": "LightGBM (new)",
     "A Macro F1": f1_xgb, "B Macro F1": f1_lgbm,
     "A Mean AOD": aod_xgb, "B Mean AOD": aod_lgbm,
     "Delta AOD": aod_lgbm - aod_xgb,
     "A Sarcasm AOD": get_sarc(audit_xgb, "AIF360 AOD"),
     "B Sarcasm AOD": get_sarc(audit_lgbm, "AIF360 AOD"),
     "Delta Sarcasm AOD": get_sarc(audit_lgbm, "AIF360 AOD") - get_sarc(audit_xgb, "AIF360 AOD"),
     "Fairer Model": "XGBoost" if aod_xgb < aod_lgbm else "LightGBM",
     "McNemar p": np.nan},
])

print("="*100)
print("TABLE 8 — MODEL DESIGN AS A FAIRNESS VARIABLE")
print("="*100)
print(table8.round(4).to_string(index=False))

save_result_table(table8, "Table8_Model_Design")
print("\nNovelty 2 complete.")

TABLE 8 — MODEL DESIGN AS A FAIRNESS VARIABLE
            Comparison           Factor Varied                     Model A            Model B  A Macro F1  B Macro F1  A Mean AOD  B Mean AOD  Delta AOD  A Sarcasm AOD  B Sarcasm AOD  Delta Sarcasm AOD        Fairer Model  McNemar p
Part A — Loss function  log loss vs hinge loss Logistic Regression (Exp 1) Linear SVM (Exp 2)      0.5672      0.5648      0.0312      0.0329     0.0017         0.0571         0.0583             0.0012 Logistic Regression     0.0018
  Part B — Tree growth level-wise vs leaf-wise             XGBoost (Exp 5)     LightGBM (new)      0.4519      0.5301      0.0336      0.0194    -0.0142        -0.0705        -0.0302             0.0404            LightGBM        NaN
  saved table -> Table8_Model_Design.csv

Novelty 2 complete.
